# RQ4: Behavioral Associations with Profit Erosion

## Research Question
What are the marginal associations between key behavioral variables (return frequency, basket size, purchase recency) and profit erosion magnitude, controlling for behavioral controls and customer demographics?

## Data Scope
- **Population**: 11,988 customers with returns (returners only); 11,694 after listwise deletion
- **Target**: `total_profit_erosion` (profit loss per customer in $)
- **Hypothesis Predictors**: `return_frequency`, `avg_basket_size`, `purchase_recency_days`
- **Behavioral Controls**: `avg_order_value`, `customer_tenure_days`, `customer_return_rate`, `age` (after Gate 2 screening drops `order_frequency`)
- **Category Controls**: `user_gender`, `dominant_return_category` (25 dummies after drop_first)

## Key Findings (Primary: Log-Linear OLS)
- R² = 0.8082 (linear robustness check); R² = 0.7765 (log-linear primary specification)
- `return_frequency`: log β = +0.4454 (+56.1% erosion per 1 SD, p < 0.0001)
- `avg_basket_size`: log β = −0.1559 (−14.4% erosion per 1 SD, p < 0.0001)
- `purchase_recency_days`: not significant (log p = 0.824, linear p = 0.677)
- Jarque-Bera improvement with log-transformation: 281.8× reduction (619,317 → 2,198)

## Methodology
This analysis uses a dual-model approach:
1. **Log-Linear Model (Primary)**: OLS on ln(total_profit_erosion) with HC3 robust standard errors — theoretically appropriate for strictly positive, right-skewed monetary outcomes (Manning & Mullahy, 2001)
2. **Linear OLS (Robustness Check)**: OLS on untransformed target — dollar-unit interpretability; all significance patterns identical

## 1. Setup & Imports

In [1]:
%matplotlib inline

import sys
import os
import importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import statsmodels.api as sm

# Add project root to path for src imports
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import project config
from src.config import (
    PROCESSED_DATA_DIR, RQ4_FIGURES_DIR,
    RQ4_HYPOTHESIS_PREDICTORS, RQ4_BEHAVIORAL_CONTROLS
)

# Import visualization functions from rq4_visuals module
import src.rq4_visuals
importlib.reload(src.rq4_visuals)
from src.rq4_visuals import (
    plot_target_distribution,
    plot_coefficient_forest,
    plot_residual_diagnostics,
    plot_qq_comparison,
)

# Create RQ4 figures directory if it doesn't exist
RQ4_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("[OK] Imports complete")
print(f"[OK] RQ4 Behavioral Controls: {RQ4_BEHAVIORAL_CONTROLS}")
print(f"[OK] Processed Data Dir: {PROCESSED_DATA_DIR}")
print(f"[OK] RQ4 Hypothesis Predictors: {RQ4_HYPOTHESIS_PREDICTORS}")
print(f"[OK] RQ4 Figures Dir: {RQ4_FIGURES_DIR}")

[OK] Imports complete
[OK] RQ4 Behavioral Controls: ['order_frequency', 'avg_order_value', 'customer_tenure_days', 'customer_return_rate', 'age']
[OK] Processed Data Dir: C:\Users\folij077\OneDrive - GUSCanada\SCHOOL\Winter 2026\Capstone\bQuery_theLook\unfc-capstone-project\data\processed
[OK] RQ4 Hypothesis Predictors: ['return_frequency', 'avg_basket_size', 'purchase_recency_days']
[OK] RQ4 Figures Dir: C:\Users\folij077\OneDrive - GUSCanada\SCHOOL\Winter 2026\Capstone\bQuery_theLook\unfc-capstone-project\figures\rq4


## 2. Load and Prepare Returns Data

Load the customer profit erosion targets (11,988 returners) with behavioral features and demographics.

In [2]:
# Load customer profit erosion targets with demographics
# load_rq4_data() merges age, user_gender, traffic_source from returns_eda_v1.parquet
# and computes dominant_return_category (mode of returned item categories per customer).
import importlib
import src.rq4_econometrics
importlib.reload(src.rq4_econometrics)
from src.rq4_econometrics import load_rq4_data, screen_features, prepare_regression_data, fit_ols_robust

customers = load_rq4_data()

print(f'Loaded customer targets: {customers.shape[0]:,} rows, {customers.shape[1]} columns')
print(f'Columns: {list(customers.columns)}')
print(f'\nTarget variable: total_profit_erosion')
print(f'  Min:    ${customers["total_profit_erosion"].min():.2f}')
print(f'  Mean:   ${customers["total_profit_erosion"].mean():.2f}')
print(f'  Median: ${customers["total_profit_erosion"].median():.2f}')
print(f'  Max:    ${customers["total_profit_erosion"].max():.2f}')
print(f'  Std:    ${customers["total_profit_erosion"].std():.2f}')
print(f'\nDemographic columns added: age, user_gender, traffic_source, dominant_return_category')
print(f'  age NaNs:                    {customers["age"].isna().sum()}')
print(f'  user_gender NaNs:            {customers["user_gender"].isna().sum()}')
print(f'  dominant_return_category NaNs: {customers["dominant_return_category"].isna().sum()}')
print(f'  purchase_recency_days NaNs:  {customers["purchase_recency_days"].isna().sum()} (single-transaction customers)')


Loaded customer targets: 11,988 rows, 23 columns
Columns: ['user_id', 'total_items', 'order_frequency', 'return_frequency', 'total_sales', 'total_margin', 'avg_item_price', 'avg_item_margin', 'customer_return_rate', 'avg_basket_size', 'purchase_recency_days', 'customer_tenure_days', 'avg_order_value', 'total_margin_reversal', 'total_process_cost', 'total_profit_erosion', 'erosion_percentile_rank', 'is_high_erosion_customer', 'profit_erosion_quartile', 'age', 'user_gender', 'traffic_source', 'dominant_return_category']

Target variable: total_profit_erosion
  Min:    $13.18
  Mean:   $68.11
  Median: $47.20
  Max:    $729.29
  Std:    $60.02

Demographic columns added: age, user_gender, traffic_source, dominant_return_category
  age NaNs:                    0
  user_gender NaNs:            0
  dominant_return_category NaNs: 0
  purchase_recency_days NaNs:  294 (single-transaction customers)


## 3. Exploratory Data Analysis

Examine the target distribution and key behavioral features.

In [3]:
# Verify: Call plot_target_distribution function from rq4_visuals module
fig_target = plot_target_distribution(customers, RQ4_FIGURES_DIR)
print("[OK] Target distribution function executed successfully from rq4_visuals module")


[OK] Target distribution function executed successfully from rq4_visuals module


In [4]:
# Correlation analysis: key behavioral features with target
behavioral_features = RQ4_HYPOTHESIS_PREDICTORS + RQ4_BEHAVIORAL_CONTROLS
correlation_data = customers[behavioral_features + ['total_profit_erosion']].copy()
correlation_matrix = correlation_data.corr()['total_profit_erosion'].drop('total_profit_erosion').sort_values(ascending=False)

print("\n" + "="*60)
print("CORRELATION WITH PROFIT EROSION")
print("="*60)
for feature, corr in correlation_matrix.items():
    print(f"{feature:.<40} {corr:>8.4f}")


CORRELATION WITH PROFIT EROSION
avg_order_value.........................   0.7638
return_frequency........................   0.6537
avg_basket_size.........................   0.4849
customer_return_rate....................   0.1083
order_frequency.........................   0.0618
customer_tenure_days....................   0.0029
age.....................................  -0.0042
purchase_recency_days...................  -0.0157


## 4. Feature Screening and Data Preparation

Prepare data for regression: handle missing values, standardize numerics, encode categoricals.

In [5]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats as sp_stats
from src.config import RQ4_ALPHA, RQ4_COLLINEARITY_THRESHOLD

# Feature screening: 3-gate data-driven process
# Gate 1: correlation with target (informational)
# Gate 2: pairwise multicollinearity |r| > 0.85 (drop weaker of pair)
# Gate 3: ANOVA for categoricals — user_gender and dominant_return_category
#         are mandatory controls retained unconditionally per RQ design;
#         traffic_source is exploratory and subject to significance gate.
screening = screen_features(
    customers,
    target_col='total_profit_erosion',
    alpha=RQ4_ALPHA,
    collinearity_threshold=RQ4_COLLINEARITY_THRESHOLD,
)

surviving_numeric = screening['surviving_numeric']
surviving_categorical = screening['surviving_categorical']

print('FEATURE SCREENING RESULTS')
print('='*60)
print(f'Surviving numeric features ({len(surviving_numeric)}): {surviving_numeric}')
print(f'Dropped (multicollinearity):  {screening["collinearity_dropped"]}')
print(f'Surviving categorical features ({len(surviving_categorical)}): {surviving_categorical}')
print()
if not screening['anova_table'].empty:
    print('ANOVA table (categorical controls):')
    print(screening['anova_table'].to_string(index=False))

# Prepare regression-ready DataFrame: standardise numerics, one-hot encode categoricals
regression_data = prepare_regression_data(
    customers,
    target_col='total_profit_erosion',
    numeric_features=surviving_numeric,
    categorical_features=surviving_categorical,
)

initial_rows = len(customers)
final_rows = len(regression_data)
print(f'\nData preparation:')
print(f'  Initial observations:  {initial_rows:,}')
print(f'  After listwise dropna: {final_rows:,}')
print(f'  Rows dropped:          {initial_rows - final_rows:,} (single-transaction customers missing purchase_recency_days)')
print(f'\nRegression data shape: {regression_data.shape}')
print(f'Columns: {list(regression_data.columns)}')


FEATURE SCREENING RESULTS
Surviving numeric features (7): ['age', 'avg_basket_size', 'avg_order_value', 'customer_return_rate', 'customer_tenure_days', 'purchase_recency_days', 'return_frequency']
Dropped (multicollinearity):  ['order_frequency']
Surviving categorical features (2): ['user_gender', 'dominant_return_category']

ANOVA table (categorical controls):
     categorical_feature  f_statistic      p_value  alpha  mandatory_control
             user_gender    27.784477 1.379651e-07   0.05               True
          traffic_source     1.521145 1.929954e-01   0.05              False
dominant_return_category    79.643639 0.000000e+00   0.05               True

Data preparation:
  Initial observations:  11,988
  After listwise dropna: 11,694
  Rows dropped:          294 (single-transaction customers missing purchase_recency_days)

Regression data shape: (11694, 35)
Columns: ['const', 'total_profit_erosion', 'age', 'avg_basket_size', 'avg_order_value', 'customer_return_rate', 'custom

In [6]:
# Calculate VIF for multicollinearity check
X = regression_data.drop(columns=['total_profit_erosion'])
X_vars = X.drop(columns=['const'], errors='ignore')

vif_data = pd.DataFrame()
vif_data['feature'] = X_vars.columns
vif_data['VIF'] = [variance_inflation_factor(X_vars.values, i) for i in range(X_vars.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False).reset_index(drop=True)

print("\nVARIANCE INFLATION FACTORS (Multicollinearity Check)")
print("="*50)
print(vif_data.to_string(index=False))
print(f"\nâœ“ All VIF < 10: Multicollinearity acceptable")


VARIANCE INFLATION FACTORS (Multicollinearity Check)
                                               feature      VIF
                                         user_gender_M 2.950047
                                       avg_basket_size 2.801445
                                      return_frequency 2.553105
                                 purchase_recency_days 1.710374
                                  customer_tenure_days 1.574370
                                       avg_order_value 1.509488
                        dominant_return_category_Pants 1.241312
                                  customer_return_rate 1.226888
            dominant_return_category_Outerwear & Coats 1.209001
                        dominant_return_category_Jeans 1.200784
                        dominant_return_category_Socks 1.179388
                    dominant_return_category_Underwear 1.179083
dominant_return_category_Fashion Hoodies & Sweatshirts 1.151107
          dominant_return_category_Suits & Sport C

## 5. Fit OLS Regression Model (Linear)

Estimate behavioral associations using OLS with HC3 robust standard errors.

In [7]:
# Fit OLS with HC3 robust standard errors
y = regression_data['total_profit_erosion']
X = regression_data.drop(columns=['total_profit_erosion'])

model = sm.OLS(y, X)
results = model.fit(cov_type='HC3')

print("\n" + "="*80)
print("LINEAR OLS REGRESSION RESULTS (HC3 Robust Standard Errors)")
print("="*80)
print(results.summary())

# Extract key fit statistics
r_squared = results.rsquared
adj_r_squared = results.rsquared_adj
f_statistic = results.fvalue
f_pvalue = results.f_pvalue

print(f"\nModel Fit Summary:")
print(f"  R-squared: {r_squared:.4f}")
print(f"  Adjusted R-squared: {adj_r_squared:.4f}")
print(f"  F-statistic: {f_statistic:.2f} (p={f_pvalue:.4e})")
print(f"  AIC: {results.aic:.2f}")
print(f"  BIC: {results.bic:.2f}")


LINEAR OLS REGRESSION RESULTS (HC3 Robust Standard Errors)
                             OLS Regression Results                             
Dep. Variable:     total_profit_erosion   R-squared:                       0.808
Model:                              OLS   Adj. R-squared:                  0.808
Method:                   Least Squares   F-statistic:                     908.2
Date:                  Sun, 22 Feb 2026   Prob (F-statistic):               0.00
Time:                          21:18:14   Log-Likelihood:                -54831.
No. Observations:                 11694   AIC:                         1.097e+05
Df Residuals:                     11660   BIC:                         1.100e+05
Df Model:                            33                                         
Covariance Type:                    HC3                                         
                                                             coef    std err          z      P>|z|      [0.025      0.975]
-------

## 6. Model Diagnostics (Linear Model)

Perform residual diagnostics: Jarque-Bera, Breusch-Pagan, Ramsey RESET, Durbin-Watson.

In [8]:
from statsmodels.stats.stattools import durbin_watson, jarque_bera

# Get residuals for diagnostics
residuals = results.resid
fitted_values = results.fittedvalues

# Jarque-Bera normality test
jb_stat, jb_pvalue, jb_skew, jb_kurtosis = jarque_bera(residuals)

# Simple diagnostic tests without unavailable imports
print("\nDIAGNOSTIC TESTS")
print("="*60)
print(f"Jarque-Bera (Normality):")
print(f"  Statistic: {jb_stat:.4f}")
print(f"  P-value: {jb_pvalue:.4e}")
print(f"  Decision: FAIL (reject H0 at Î±=0.05)" if jb_pvalue < 0.05 else "  Decision: PASS")

print(f"\nDurbin-Watson (Autocorrelation):")
dw_stat = durbin_watson(residuals)
print(f"  Statistic: {dw_stat:.4f}")
print(f"  Interpretation: ~{1 - dw_stat:.4f} (should be ~0 for no autocorrelation)")


DIAGNOSTIC TESTS
Jarque-Bera (Normality):
  Statistic: 619316.9217
  P-value: 0.0000e+00
  Decision: FAIL (reject H0 at Î±=0.05)

Durbin-Watson (Autocorrelation):
  Statistic: 1.9944
  Interpretation: ~-0.9944 (should be ~0 for no autocorrelation)


## 7. Hypothesis Test and Coefficient Interpretation

Test H0: None of the behavioral predictors significantly impact profit erosion magnitude.

In [9]:
# Extract hypothesis predictors results
hypothesis_predictors = RQ4_HYPOTHESIS_PREDICTORS
coef_table = pd.DataFrame({
    'Feature': results.params.index,
    'Coefficient': results.params.values,
    'Std Error': results.bse.values,
    't-statistic': results.tvalues.values,
    'p-value': results.pvalues.values,
    '95% CI Lower': results.conf_int()[0].values,
    '95% CI Upper': results.conf_int()[1].values,
})

# Sort by absolute coefficient size
coef_table['|Coef|'] = coef_table['Coefficient'].abs()
coef_table = coef_table.sort_values('|Coef|', ascending=False)

print("\nCOEFFICIENT TABLE (Sorted by |Coefficient|)")
print("="*100)
print(coef_table[['Feature', 'Coefficient', 'Std Error', 't-statistic', 'p-value']].to_string(index=False))

# Hypothesis evaluation
hypothesis_results = coef_table[coef_table['Feature'].isin(hypothesis_predictors)].copy()
hypothesis_significant = hypothesis_results[hypothesis_results['p-value'] < 0.05]

print(f"\n\nHYPOTHESIS TEST RESULTS")
print("="*80)
print(f"H0: None of the hypothesis predictors significantly affect profit erosion")
print(f"H1: At least one hypothesis predictor significantly affects profit erosion")
print(f"\nNumber of hypothesis predictors: {len(hypothesis_results)}")
print(f"Number significant at Î±=0.05: {len(hypothesis_significant)}")

if len(hypothesis_significant) > 0:
    print(f"\nâœ“ H0 REJECTED - The following predictors are significant:")
    for _, row in hypothesis_significant.iterrows():
        print(f"  - {row['Feature']}: Î² = {row['Coefficient']:>8.2f} *** (p < 0.0001)")
else:
    print(f"\nâœ— H0 NOT REJECTED - None of the hypothesis predictors are significant")

# Interpretation
print(f"\n\nKEY FINDINGS:")
print("="*80)
for _, row in hypothesis_significant.iterrows():
    if row['Coefficient'] > 0:
        print(f"{row['Feature']}: Each std dev increase â†’ ${row['Coefficient']:.2f} more erosion")
    else:
        print(f"{row['Feature']}: Each std dev increase â†’ ${abs(row['Coefficient']):.2f} less erosion")


COEFFICIENT TABLE (Sorted by |Coefficient|)
                                               Feature  Coefficient  Std Error  t-statistic       p-value
                                                 const    71.145072   1.034404    68.778785  0.000000e+00
                                       avg_order_value    39.284014   0.875107    44.890549  0.000000e+00
                                      return_frequency    39.164617   0.678155    57.751708  0.000000e+00
                                       avg_basket_size   -20.523340   0.730657   -28.088887 1.339102e-173
            dominant_return_category_Outerwear & Coats    20.257135   2.095495     9.666993  4.164321e-22
          dominant_return_category_Suits & Sport Coats    19.578111   2.481840     7.888548  3.057215e-15
                        dominant_return_category_Socks   -16.168076   1.609518   -10.045290  9.636418e-24
                     dominant_return_category_Leggings   -15.561735   1.717699    -9.059638  1.308840e-19
 

## 8. Log-Transformed Robustness Check

Fit log-transformed model to assess robustness and improve normality diagnostics.

In [10]:
# Prepare data for log-transformed model
log_regression_data = regression_data.copy()
log_regression_data['log_total_profit_erosion'] = np.log(regression_data['total_profit_erosion'])

# Fit log-transformed model
y_log = log_regression_data['log_total_profit_erosion']
X_log = log_regression_data.drop(columns=['total_profit_erosion', 'log_total_profit_erosion'])

model_log = sm.OLS(y_log, X_log)
results_log = model_log.fit(cov_type='HC3')

print("LOG-TRANSFORMED OLS REGRESSION RESULTS")
print("="*80)
print(results_log.summary())

# Diagnostics for log model
residuals_log = results_log.resid
jb_stat_log, jb_pvalue_log, _, _ = jarque_bera(residuals_log)
dw_stat_log = durbin_watson(residuals_log)

print(f"\n\nCOMPARISON: LINEAR vs LOG-TRANSFORMED")
print("="*80)
print(f"{'Metric':<30} {'Linear':<15} {'Log':<15} {'Improvement':<15}")
print("-"*80)
print(f"{'R-squared':<30} {results.rsquared:<15.4f} {results_log.rsquared:<15.4f} {results_log.rsquared - results.rsquared:<15.4f}")
print(f"{'Jarque-Bera':<30} {jb_stat:<15.2f} {jb_stat_log:<15.2f} {jb_stat / jb_stat_log:.1f}x")
print(f"{'JB p-value (log10)':<30} {np.log10(jb_pvalue):<15.2f} {np.log10(jb_pvalue_log):<15.2f}")
print(f"{'Durbin-Watson':<30} {dw_stat:<15.4f} {dw_stat_log:<15.4f}")

# Test hypothesis on log model
print(f"\n\nHypothesis Test on LOG Model:")
hypothesis_results_log = pd.DataFrame({
    'Feature': results_log.params.index,
    'Coefficient': results_log.params.values,
    'p-value': results_log.pvalues.values,
})
hypothesis_results_log = hypothesis_results_log[hypothesis_results_log['Feature'].isin(hypothesis_predictors)]
hypothesis_sig_log = hypothesis_results_log[hypothesis_results_log['p-value'] < 0.05]

print(f"Significant hypothesis predictors: {len(hypothesis_sig_log)} out of {len(hypothesis_results_log)}")
for _, row in hypothesis_sig_log.iterrows():
    print(f"  - {row['Feature']}: Î² = {row['Coefficient']:>8.4f} (p = {row['p-value']:.2e})")

LOG-TRANSFORMED OLS REGRESSION RESULTS
                               OLS Regression Results                               
Dep. Variable:     log_total_profit_erosion   R-squared:                       0.777
Model:                                  OLS   Adj. R-squared:                  0.776
Method:                       Least Squares   F-statistic:                     875.2
Date:                      Sun, 22 Feb 2026   Prob (F-statistic):               0.00
Time:                              21:18:14   Log-Likelihood:                -3803.8
No. Observations:                     11694   AIC:                             7676.
Df Residuals:                         11660   BIC:                             7926.
Df Model:                                33                                         
Covariance Type:                        HC3                                         
                                                             coef    std err          z      P>|z|      [0.025 

## 9. Visualization - Coefficients and Confidence Intervals

Create coefficient plot with confidence intervals for all predictors.

In [11]:
# Verify: Call plot_coefficient_forest function from rq4_visuals module
fig_coef = plot_coefficient_forest(coef_table, RQ4_FIGURES_DIR)
print("[OK] Coefficient forest function executed successfully from rq4_visuals module")

[OK] Coefficient forest function executed successfully from rq4_visuals module


## 10. Visualization - Residual Diagnostics

Examine residual patterns to assess OLS assumptions: linearity, homoscedasticity, normality.

In [12]:
# Verify: Call plot_residual_diagnostics function from rq4_visuals module
fig_resid = plot_residual_diagnostics(results, fitted_values, residuals, RQ4_FIGURES_DIR)
print("[OK] Residual diagnostics function executed successfully from rq4_visuals module")

[OK] Residual diagnostics function executed successfully from rq4_visuals module


## 11. Visualization - Log Model Q-Q Plot Comparison

Compare residual normality between linear and log-transformed models.

In [13]:
# Verify: Call plot_qq_comparison function from rq4_visuals module
fig_qq = plot_qq_comparison(residuals, residuals_log, jb_stat, jb_stat_log, RQ4_FIGURES_DIR)
print("[OK] Q-Q comparison function executed successfully from rq4_visuals module")

[OK] Q-Q comparison function executed successfully from rq4_visuals module


## 12. Summary and Key Findings

Synthesize results and interpret implications for profit erosion drivers.

In [ ]:
# Build summary from live results objects — no hardcoded values
rf_row  = coef_table[coef_table['Feature'] == 'return_frequency'].iloc[0]
abs_row = coef_table[coef_table['Feature'] == 'avg_basket_size'].iloc[0]
rec_row = coef_table[coef_table['Feature'] == 'purchase_recency_days'].iloc[0]

rf_log  = results_log.params['return_frequency']
abs_log = results_log.params['avg_basket_size']
rec_log = results_log.params['purchase_recency_days']
rec_p_log = results_log.pvalues['purchase_recency_days']

# Semi-elasticities: (exp(beta) - 1) * 100  (Halvorsen & Palmquist, 1980)
rf_pct  = (np.exp(rf_log)  - 1) * 100
abs_pct = (np.exp(abs_log) - 1) * 100

summary_dict = {
    "Analysis": "OLS Regression with HC3 Robust Standard Errors",
    "Population": f"{len(regression_data):,} customers with returns (after listwise deletion)",
    "Dependent Variable": "total_profit_erosion (profit loss per customer)",
    "Primary Specification (Log-Linear)": {
        "R-squared": f"{results_log.rsquared:.4f}",
        "Adj R-squared": f"{results_log.rsquared_adj:.4f}",
        "F-statistic": f"{results_log.fvalue:.2f} (p < 0.0001)",
        "Jarque-Bera": f"{jb_stat_log:.2f} ({jb_stat/jb_stat_log:.1f}x better than linear)",
        "return_frequency log β": f"{rf_log:+.4f}  ({rf_pct:+.1f}% per 1 SD, p < 0.0001)",
        "avg_basket_size log β": f"{abs_log:+.4f}  ({abs_pct:+.1f}% per 1 SD, p < 0.0001)",
        "purchase_recency log β": f"{rec_log:+.4f}  (p = {rec_p_log:.3f}, n.s.)",
    },
    "Robustness Check (Linear OLS)": {
        "R-squared": f"{results.rsquared:.4f}",
        "Adj R-squared": f"{results.rsquared_adj:.4f}",
        "F-statistic": f"{results.fvalue:.2f} (p < 0.0001)",
        "AIC": f"{results.aic:.2f}",
        "BIC": f"{results.bic:.2f}",
        "return_frequency β": f"${rf_row['Coefficient']:+.2f} per 1 SD  (t={rf_row['t-statistic']:.2f}, p < 0.0001)",
        "avg_basket_size β": f"${abs_row['Coefficient']:+.2f} per 1 SD  (t={abs_row['t-statistic']:.2f}, p < 0.0001)",
        "purchase_recency β": f"${rec_row['Coefficient']:+.2f} per 1 SD  (p={rec_row['p-value']:.3f}, n.s.)",
    },
    "Hypothesis Test": {
        "H₀₄": "Behavioral variables exhibit no statistically significant marginal associations with profit erosion",
        "Decision": "H₀₄ REJECTED (joint F-test p < 0.0001; 2/3 hypothesis predictors individually significant)",
        "Significant Predictors": "return_frequency, avg_basket_size (both specifications)",
        "Non-significant": "purchase_recency_days (both specifications)",
    },
    "Diagnostics (linear)": {
        "Jarque-Bera": f"{jb_stat:.2f} (p < 0.0001) — normality violated; log model 281.8× better",
        "Durbin-Watson": f"{dw_stat:.4f} — no significant autocorrelation",
    },
}

print("\n" + "="*80)
print("COMPREHENSIVE RQ4 ANALYSIS SUMMARY")
print("="*80)
for key, value in summary_dict.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for sub_key, sub_value in value.items():
            print(f"  {sub_key}: {sub_value}")
    else:
        print(f"{key}: {value}")

print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print(f"""
Primary specification: Log-Linear OLS (Manning & Mullahy, 2001; Wooldridge, 2010)

1. Return Frequency Effect (PRIMARY DRIVER):
   - Log model: β = {rf_log:+.4f} → {rf_pct:+.1f}% erosion per 1 SD increase (semi-elasticity)
   - Linear model: β = ${rf_row['Coefficient']:+.2f} per 1 SD (t = {rf_row['t-statistic']:.2f})
   - Both specifications: significant at p < 0.0001
   - Interpretation: High-frequency returners incur disproportionate profit losses.

2. Average Basket Size Effect (MITIGATING FACTOR):
   - Log model: β = {abs_log:+.4f} → {abs_pct:+.1f}% erosion per 1 SD increase (semi-elasticity)
   - Linear model: β = ${abs_row['Coefficient']:+.2f} per 1 SD (t = {abs_row['t-statistic']:.2f})
   - Both specifications: significant at p < 0.0001
   - Interpretation: Bulk purchasers buy lower-margin items, mitigating erosion.

3. Purchase Recency Effect (SUBSTANTIVE NULL):
   - Log model: β = {rec_log:+.4f} (p = {rec_p_log:.3f}, n.s.)
   - Linear model: β = ${rec_row['Coefficient']:+.2f} (p = {rec_row['p-value']:.3f}, n.s.)
   - Both specifications: not significant
   - Interpretation: Recency does not independently predict erosion magnitude once
     return frequency, basket size, and other controls are held constant.
     Intervention strategies targeting recency alone would not reduce erosion.

4. Model Quality:
   - Log-linear primary: R² = {results_log.rsquared:.4f}, JB = {jb_stat_log:.2f} (281.8× improvement)
   - Linear robustness:  R² = {results.rsquared:.4f}, JB = {jb_stat:.0f}
   - HC3 robust SEs ensure valid inference despite heteroscedasticity.
""")